# Store Data - Simple Baseline

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


## Imports

In [ ]:
import mlflow
import torch
from torch import Tensor
from torch.utils.data import DataLoader, Subset

from common.git import get_branch, get_sha
from common.model_registry import TRACKING_URI
from time_series.main_store_sales_transformer import StoreSalesTransformer
from time_series.store_sales import MSLELoss, StoreData, Trainer

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

## mlflow Setup

In [ ]:
mlflow.pytorch.autolog()  # pyright: ignore[reportPrivateImportUsage]
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("StoreSales_TransformerBasic")
mlflow.set_experiment_tags(
    {
        "dataset": "store-sales-kaggle",
        "task": "time-series-forecasting",
        "loss": "RMSLE",
        "prediction_horizon_days": "16",
    }
)



# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData(dtype=torch.float32)
store_data_loader = DataLoader(store_data, batch_size=10)
store_data.train

# Model Design

- Input data is essentially `date x store_nbr x family`, which is easily tensor-able. The overall data will be `batch, date_len, store_nbr, family`. 
  - Remember to use `batch_first=True` for any LSTM or whatever models. Apparently having the batch dimension be second is an old convention.
- We need to predict 2017-08-16 to 2017-08-31, so 16 day prediction window
- Input window length will be a parameter, sent to the dataset and model
- loss is Root-Mean-Squared-Logarithmic-Error (given by problem)
  - $\text{RMSE}(\log(1+\hat{y}), \log(1+y))$

In [ ]:
model = StoreSalesTransformer(
    n_stores=store_data.stores.shape[0],
    n_families=store_data.families.size,
    n_output_steps=store_data.output_lags,
)
model

In [ ]:
batch_X, batch_y = next(iter(store_data_loader))
model(batch_X).shape

# Training

In [ ]:
epochs = 500
save_every_n_epochs = 50
learning_rate = 1e-3
split_fraction = 0.9
batch_size = 128

split_loc = int(len(store_data) * split_fraction)

train_data = Subset(store_data, range(0, split_loc))
val_data = Subset(store_data, range(split_loc, len(store_data)))

train_loader = DataLoader[Tensor](train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)

In [ ]:
with mlflow.start_run(
    # run_name="initial_training",
    tags={
        "architecture": "transformer",
        "input_window_days": str(store_data.window_lags),
        "output_window_days": str(store_data.output_lags),
        "device": str(device),
        "git_branch": get_branch(),
        "git_sha": get_sha(),
    },
):
    mlflow.log_params(
        {
            "epochs": epochs,
            "learning_rate": learning_rate,
            "split_fraction": split_fraction,
            "batch_size": batch_size,
        }
    )
    train_loader = DataLoader[Tensor](train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)
    trainer = Trainer(
        model=model,
        device=device,
        train_loader=train_loader,
        val_loader=val_loader,
        learning_rate=learning_rate,
        loss_func=MSLELoss(),
    )
    trainer.train(epochs, save_every_n_epochs=save_every_n_epochs)

# Scrap

In [ ]:

# # Batch Size Sweep

# with mlflow.start_run(
#     run_name="batch_size_sweep",
#     tags={
#         "architecture": "transformer",
#         "input_window_days": str(store_data.window_lags),
#         "output_window_days": str(store_data.output_lags),
#         "device": str(device),
#         "git_branch": get_branch(),
#         "git_sha": get_sha(),
#     },
# ) as mlflow_run:
#     mlflow.log_params(
#         {
#             "epochs": epochs,
#             "learning_rate": learning_rate,
#             "split_fraction": split_fraction,
#             # "batch_size": batch_size,
#         }
#     )
#     for batch_size in [64, 128, 256, 512]:
#         with mlflow.start_run(nested=True) as subrun:
#             mlflow.log_param("batch_size", batch_size)
#             train_loader = DataLoader[Tensor](
#                 train_data, batch_size=batch_size, shuffle=True
#             )
#             val_loader = DataLoader[Tensor](val_data, batch_size=batch_size)
#             trainer = Trainer(model, train_loader=train_loader, val_loader=val_loader)
#             results_df = trainer.train(epochs)
#             results_df.plot(sharex=True, subplots=True, figsize=(8, 12))